In [1]:
import pandas as pd
from pathlib import Path

airbnb = pd.read_csv("../../filtered_listings_reviews-mo.csv", low_memory=False)
airbnb.head()

,listing_id,review_id,listing_name,description,review,rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,neighborhood,price,availability_30,availability_60,availability_90,availability_365,availability_eoy
0,29059,55332,Lovely studio Quartier Latin,"CITQ 267153 Lovely studio with 1 closed room, ...",This apt was soooo adorable! Centrally located...,4.69,4.79,4.64,4.82,4.79,4.82,4.68,Ville-Marie,134.0,7,21,48,312,61
1,29059,59988,Lovely studio Quartier Latin,"CITQ 267153 Lovely studio with 1 closed room, ...",I had a wonderful stay at Maryline's lovely an...,4.69,4.79,4.64,4.82,4.79,4.82,4.68,Ville-Marie,134.0,7,21,48,312,61
2,29059,61088,Lovely studio Quartier Latin,"CITQ 267153 Lovely studio with 1 closed room, ...",Both my husband and I had a wonderful experien...,4.69,4.79,4.64,4.82,4.79,4.82,4.68,Ville-Marie,134.0,7,21,48,312,61
3,29059,386009,Lovely studio Quartier Latin,"CITQ 267153 Lovely studio with 1 closed room, ...",This was our first trip to Montreal and we had...,4.69,4.79,4.64,4.82,4.79,4.82,4.68,Ville-Marie,134.0,7,21,48,312,61
4,29059,399065,Lovely studio Quartier Latin,"CITQ 267153 Lovely studio with 1 closed room, ...",We stayed in Maryline's studio for a long week...,4.69,4.79,4.64,4.82,4.79,4.82,4.68,Ville-Marie,134.0,7,21,48,312,61


In [2]:
import numpy as np

def load_glove(path, dim):
    vectors = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.rstrip().split(" ")
            word = parts[0]
            vals = np.asarray(parts[1:], dtype=np.float32)
            if vals.shape[0] != dim:
                continue
            vectors[word] = vals
    return vectors

glove = load_glove("../../glove.6B/glove.6B.100d.txt", dim=100)


In [3]:
import re

token_re = re.compile(r"[a-z]+(?:'[a-z]+)?")

def text_to_glove_avg(text, glove, dim):
    
    tokens = token_re.findall(text.lower())
    vecs = [glove[t] for t in tokens if t in glove]

    if not vecs:
        return np.zeros(dim, dtype=np.float32)

    return np.mean(vecs, axis=0).astype(np.float32)


In [4]:
DIM = 100

airbnb["desc_emb"] = airbnb["description"].apply(lambda x: text_to_glove_avg(x, glove, DIM))
airbnb["review_emb"] = airbnb["review"].apply(lambda x: text_to_glove_avg(x, glove, DIM))
airbnb.head()


,listing_id,review_id,listing_name,description,review,rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,...,review_scores_value,neighborhood,price,availability_30,availability_60,availability_90,availability_365,availability_eoy,desc_emb,review_emb
0,29059,55332,Lovely studio Quartier Latin,"CITQ 267153 Lovely studio with 1 closed room, ...",This apt was soooo adorable! Centrally located...,4.69,4.79,4.64,4.82,4.79,...,4.68,Ville-Marie,134.0,7,21,48,312,61,"[-0.16316804, 0.18909205, 0.102388084, -0.0856...","[-0.25262606, 0.22221571, 0.28349322, -0.09773..."
1,29059,59988,Lovely studio Quartier Latin,"CITQ 267153 Lovely studio with 1 closed room, ...",I had a wonderful stay at Maryline's lovely an...,4.69,4.79,4.64,4.82,4.79,...,4.68,Ville-Marie,134.0,7,21,48,312,61,"[-0.16316804, 0.18909205, 0.102388084, -0.0856...","[-0.18417816, 0.17882091, 0.2914532, -0.193029..."
2,29059,61088,Lovely studio Quartier Latin,"CITQ 267153 Lovely studio with 1 closed room, ...",Both my husband and I had a wonderful experien...,4.69,4.79,4.64,4.82,4.79,...,4.68,Ville-Marie,134.0,7,21,48,312,61,"[-0.16316804, 0.18909205, 0.102388084, -0.0856...","[-0.10504398, 0.20931998, 0.29978752, -0.19985..."
3,29059,386009,Lovely studio Quartier Latin,"CITQ 267153 Lovely studio with 1 closed room, ...",This was our first trip to Montreal and we had...,4.69,4.79,4.64,4.82,4.79,...,4.68,Ville-Marie,134.0,7,21,48,312,61,"[-0.16316804, 0.18909205, 0.102388084, -0.0856...","[-0.15261658, 0.22745192, 0.36459345, -0.17985..."
4,29059,399065,Lovely studio Quartier Latin,"CITQ 267153 Lovely studio with 1 closed room, ...",We stayed in Maryline's studio for a long week...,4.69,4.79,4.64,4.82,4.79,...,4.68,Ville-Marie,134.0,7,21,48,312,61,"[-0.16316804, 0.18909205, 0.102388084, -0.0856...","[-0.14381, 0.11658036, 0.3006493, -0.21620901,..."


In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Fit TF-IDF on BOTH fields so weights are meaningful
corpus = pd.concat([airbnb["description"].fillna(""), airbnb["review"].fillna("")], axis=0).astype(str)

tfidf = TfidfVectorizer(lowercase=True, token_pattern=r"[a-z]+(?:'[a-z]+)?", min_df=2)
tfidf.fit(corpus)

vocab = tfidf.vocabulary_
idf = tfidf.idf_

def text_to_glove_tfidf(text, glove, dim, vocab, idf):

    tokens = token_re.findall(text.lower())
    num = np.zeros(dim, dtype=np.float32)
    den = 0.0

    for t in tokens:
        if t in glove and t in vocab:
            w = float(idf[vocab[t]])  # IDF weight
            num += w * glove[t]
            den += w

    if den == 0.0:
        return np.zeros(dim, dtype=np.float32)
    return (num / den).astype(np.float32)


In [6]:
DIM = 100

airbnb["desc_emb_tfidf"] = airbnb["description"].apply(lambda x: text_to_glove_tfidf(x, glove, DIM, vocab, idf))
airbnb["review_emb_tfidf"] = airbnb["review"].apply(lambda x: text_to_glove_tfidf(x, glove, DIM, vocab, idf))
airbnb.head()

,listing_id,review_id,listing_name,description,review,rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,...,price,availability_30,availability_60,availability_90,availability_365,availability_eoy,desc_emb,review_emb,desc_emb_tfidf,review_emb_tfidf
0,29059,55332,Lovely studio Quartier Latin,"CITQ 267153 Lovely studio with 1 closed room, ...",This apt was soooo adorable! Centrally located...,4.69,4.79,4.64,4.82,4.79,...,134.0,7,21,48,312,61,"[-0.16316804, 0.18909205, 0.102388084, -0.0856...","[-0.25262606, 0.22221571, 0.28349322, -0.09773...","[-0.23479824, 0.25661486, 0.0045714267, -0.092...","[-0.25475904, 0.3200672, 0.31315228, -0.072079..."
1,29059,59988,Lovely studio Quartier Latin,"CITQ 267153 Lovely studio with 1 closed room, ...",I had a wonderful stay at Maryline's lovely an...,4.69,4.79,4.64,4.82,4.79,...,134.0,7,21,48,312,61,"[-0.16316804, 0.18909205, 0.102388084, -0.0856...","[-0.18417816, 0.17882091, 0.2914532, -0.193029...","[-0.23479824, 0.25661486, 0.0045714267, -0.092...","[-0.20613714, 0.20792075, 0.24263747, -0.12872..."
2,29059,61088,Lovely studio Quartier Latin,"CITQ 267153 Lovely studio with 1 closed room, ...",Both my husband and I had a wonderful experien...,4.69,4.79,4.64,4.82,4.79,...,134.0,7,21,48,312,61,"[-0.16316804, 0.18909205, 0.102388084, -0.0856...","[-0.10504398, 0.20931998, 0.29978752, -0.19985...","[-0.23479824, 0.25661486, 0.0045714267, -0.092...","[-0.09986156, 0.24602444, 0.25462073, -0.13582..."
3,29059,386009,Lovely studio Quartier Latin,"CITQ 267153 Lovely studio with 1 closed room, ...",This was our first trip to Montreal and we had...,4.69,4.79,4.64,4.82,4.79,...,134.0,7,21,48,312,61,"[-0.16316804, 0.18909205, 0.102388084, -0.0856...","[-0.15261658, 0.22745192, 0.36459345, -0.17985...","[-0.23479824, 0.25661486, 0.0045714267, -0.092...","[-0.13400117, 0.22437441, 0.31481773, -0.11754..."
4,29059,399065,Lovely studio Quartier Latin,"CITQ 267153 Lovely studio with 1 closed room, ...",We stayed in Maryline's studio for a long week...,4.69,4.79,4.64,4.82,4.79,...,134.0,7,21,48,312,61,"[-0.16316804, 0.18909205, 0.102388084, -0.0856...","[-0.14381, 0.11658036, 0.3006493, -0.21620901,...","[-0.23479824, 0.25661486, 0.0045714267, -0.092...","[-0.14387088, 0.11435704, 0.25361308, -0.19452..."


In [7]:
airbnb.to_csv("airbnb_glove_embeddings-ny.csv", index=False)